# 02 - Feature Engineering

This notebook creates features for price modelling:
- Residual demand (demand - RES generation)
- RES share and technology shares
- Time features (hour, day of week, season, etc.)
- Lag features for prices and demand
- Rolling window statistics
- Price spread and volatility features

## Outputs
- Feature-engineered dataset saved to `data_processed/features.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add src to path
sys.path.append('../')

from src.features import FeatureEngineer, feature_engineering_pipeline
from src.utils import load_config, plot_time_series

# Set display options
pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')
%matplotlib inline

## 1. Load Clean Data

In [ ]:
# Load configuration
config = load_config('../configs/modelling_config.yaml')

# Load clean data
data_path = Path(config.get('data.processed_path', '../data_processed'))
clean_data = pd.read_csv(data_path / 'clean_data.csv', index_col=0, parse_dates=True)

print(f"Loaded clean data: {clean_data.shape}")
print(f"Date range: {clean_data.index.min()} to {clean_data.index.max()}")
print(f"\nColumns: {clean_data.columns.tolist()}")

## 2. Create Fundamental Features

In [ ]:
# Initialize feature engineer
engineer = FeatureEngineer()

# Create residual demand
df_features = engineer.create_residual_demand(clean_data.copy())
print(f"✓ Created residual demand")

# Create RES share
df_features = engineer.create_res_share(df_features)
print(f"✓ Created RES share")

# Create generation shares
df_features = engineer.create_generation_shares(df_features)
print(f"✓ Created generation shares")

print(f"\nNew columns: {[col for col in df_features.columns if col not in clean_data.columns]}")

In [ ]:
# Visualize fundamental features
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Demand vs Residual Demand
if 'demand' in df_features.columns and 'residual_demand' in df_features.columns:
    df_features[['demand', 'residual_demand']].plot(ax=axes[0, 0], alpha=0.7)
    axes[0, 0].set_title('Demand vs Residual Demand')
    axes[0, 0].set_ylabel('MW')

# RES Share over time
if 'res_share' in df_features.columns:
    df_features['res_share'].plot(ax=axes[0, 1], alpha=0.7)
    axes[0, 1].set_title('RES Share')
    axes[0, 1].set_ylabel('Share (0-1)')

# Price vs Residual Demand scatter
if 'price' in df_features.columns and 'residual_demand' in df_features.columns:
    axes[1, 0].scatter(df_features['residual_demand'], df_features['price'], alpha=0.1, s=1)
    axes[1, 0].set_xlabel('Residual Demand (MW)')
    axes[1, 0].set_ylabel('Price (£/MWh)')
    axes[1, 0].set_title('Price vs Residual Demand')

# Price vs RES Share scatter
if 'price' in df_features.columns and 'res_share' in df_features.columns:
    axes[1, 1].scatter(df_features['res_share'], df_features['price'], alpha=0.1, s=1)
    axes[1, 1].set_xlabel('RES Share')
    axes[1, 1].set_ylabel('Price (£/MWh)')
    axes[1, 1].set_title('Price vs RES Share (Cannibalisation)')

plt.tight_layout()
plt.show()

## 3. Create Time Features

In [ ]:
# Create time features
df_features = engineer.create_time_features(df_features)
print(f"✓ Created time features")

time_features = [col for col in df_features.columns if any(x in col for x in ['hour', 'day', 'month', 'season', 'weekend', 'year', 'sin', 'cos'])]
print(f"\nTime features: {time_features}")

In [ ]:
# Visualize price patterns by time features
if 'price' in df_features.columns:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    
    # Price by hour
    if 'hour' in df_features.columns:
        df_features.groupby('hour')['price'].mean().plot(ax=axes[0, 0], marker='o')
        axes[0, 0].set_title('Average Price by Hour')
        axes[0, 0].set_xlabel('Hour of Day')
        axes[0, 0].set_ylabel('Price (£/MWh)')
        axes[0, 0].grid(True, alpha=0.3)
    
    # Price by day of week
    if 'day_of_week' in df_features.columns:
        df_features.groupby('day_of_week')['price'].mean().plot(ax=axes[0, 1], marker='o')
        axes[0, 1].set_title('Average Price by Day of Week')
        axes[0, 1].set_xlabel('Day (0=Mon, 6=Sun)')
        axes[0, 1].set_ylabel('Price (£/MWh)')
        axes[0, 1].grid(True, alpha=0.3)
    
    # Price by month
    if 'month' in df_features.columns:
        df_features.groupby('month')['price'].mean().plot(ax=axes[1, 0], marker='o')
        axes[1, 0].set_title('Average Price by Month')
        axes[1, 0].set_xlabel('Month')
        axes[1, 0].set_ylabel('Price (£/MWh)')
        axes[1, 0].grid(True, alpha=0.3)
    
    # Price by season
    if 'season' in df_features.columns:
        df_features.groupby('season')['price'].mean().plot(ax=axes[1, 1], kind='bar')
        axes[1, 1].set_title('Average Price by Season')
        axes[1, 1].set_xlabel('Season (1=Winter, 2=Spring, 3=Summer, 4=Autumn)')
        axes[1, 1].set_ylabel('Price (£/MWh)')
        axes[1, 1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

## 4. Create All Features (Lags + Rolling)

In [ ]:
# Get parameters from config
price_lags = config.get('features.price_lags', [1, 24, 168])
demand_lags = config.get('features.demand_lags', [24, 168])
rolling_windows = config.get('features.rolling_windows', [3, 24, 168])

print(f"Price lags: {price_lags}")
print(f"Demand lags: {demand_lags}")
print(f"Rolling windows: {rolling_windows}")

# Create all features
df_features = feature_engineering_pipeline(
    clean_data,
    price_lags=price_lags,
    demand_lags=demand_lags,
    rolling_windows=rolling_windows
)

print(f"\n✓ Feature engineering complete!")
print(f"  Total features: {len(df_features.columns)}")
print(f"  Shape: {df_features.shape}")

## 5. Feature Analysis

In [ ]:
# Check for missing values (expected in lag/rolling features)
missing = df_features.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if len(missing) > 0:
    print("Features with missing values:")
    print(missing.head(10))
else:
    print("✓ No missing values")

In [ ]:
# List all feature categories
feature_categories = {
    'Base': [col for col in df_features.columns if not any(x in col for x in ['lag', 'roll', 'sin', 'cos', 'hour', 'day', 'month', 'season', 'weekend', 'year'])],
    'Time': [col for col in df_features.columns if any(x in col for x in ['hour', 'day', 'month', 'season', 'weekend', 'year', 'sin', 'cos'])],
    'Lag': [col for col in df_features.columns if 'lag' in col],
    'Rolling': [col for col in df_features.columns if 'roll' in col]
}

for category, features in feature_categories.items():
    print(f"\n{category} features ({len(features)}):")
    print(f"  {features[:5]}..." if len(features) > 5 else f"  {features}")

## 6. Correlation Analysis

In [ ]:
# Select features for correlation analysis (excluding NaN rows)
df_no_nan = df_features.dropna()
print(f"Rows after dropping NaN: {len(df_no_nan)} ({len(df_no_nan)/len(df_features)*100:.1f}%)")

# Correlation with price
if 'price' in df_no_nan.columns:
    price_corr = df_no_nan.corr()['price'].sort_values(ascending=False)
    print("\nTop 15 features correlated with price:")
    print(price_corr.head(15))
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    price_corr.head(20).plot(kind='barh', ax=ax)
    ax.set_xlabel('Correlation with Price')
    ax.set_title('Top 20 Features Correlated with Price')
    plt.tight_layout()
    plt.show()

## 7. Save Feature Dataset

In [ ]:
# Save feature dataset
output_path = Path(config.get('data.processed_path', '../data_processed'))
output_file = output_path / 'features.csv'

df_features.to_csv(output_file)

print(f"\n✓ Features saved to: {output_file}")
print(f"  Shape: {df_features.shape}")
print(f"  Features: {len(df_features.columns)}")
print(f"  Date range: {df_features.index.min()} to {df_features.index.max()}")

## Summary

Feature engineering complete! The dataset now includes:
- Fundamental features (residual demand, RES shares)
- Time features (hour, day, month, season, cyclical encodings)
- Lag features for prices and demand
- Rolling window statistics
- Price spread and volatility features

**Next steps:**
- Proceed to notebook 03 for baseline revenue backcast
- Calculate historical revenue using actual prices and generation